In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

## Stochastic Differential Equation (SDE)

We simulate $N$ sample paths of a general Itô SDE using the **Euler–Maruyama** method:

$$dX_t = \mu(X_t)\,dt + \sigma(X_t)\,dW_t, \quad X_0 = x_0, \quad t \in [0, T]$$

where $W_t$ is standard Brownian motion, $\mu$ is the **drift**, and $\sigma$ is the **diffusion** coefficient.

The specific SDE used throughout is $\mu(x) = x$, $\sigma(x) = 1$:

$$dX_t = X_t\,dt + dW_t, \quad X_0 = 1$$

In [ ]:
def vectorized_euler_maruyama(mu, sigma, X0=1.0, T=1.0, n=10**4, N=10**4):
    """Simulate N paths of dX_t = mu(X_t) dt + sigma(X_t) dW_t.
    All Brownian increments are pre-allocated before the time loop."""
    dt = T / n
    X = np.full((N, n + 1), X0, dtype=float)

    # Generate all increments at once: shape (N, n)
    dW = np.sqrt(dt) * np.random.standard_normal((N, n))

    for i in range(n):
        X[:, i + 1] = X[:, i] + mu(X[:, i]) * dt + sigma(X[:, i]) * dW[:, i]

    return X

### Vectorized Euler–Maruyama

The interval $[0,T]$ is split into $n$ steps of size $\Delta t = T/n$. The update at step $i$ is:

$$X_{i+1} = X_i + \mu(X_i)\,\Delta t + \sigma(X_i)\,\Delta W_i, \quad i = 0,\ldots,n-1$$

All $N \times n$ Brownian increments are drawn **before** the loop in one NumPy call:

$$\Delta W_i \sim \mathcal{N}(0,\,\Delta t) = \sqrt{\Delta t}\cdot Z_i, \quad Z_i \overset{\text{i.i.d.}}{\sim} \mathcal{N}(0,1)$$

`dW` has shape $(N, n)$ — one increment per path per time step.

In [ ]:
def standard_euler_maruyama(mu, sigma, X0=1.0, T=1.0, n=10**4, N=10**4):
    """Same Euler-Maruyama scheme but Brownian increments are
    generated inside the loop, one time step at a time."""
    dt = T / n
    sqrt_dt = np.sqrt(dt)
    X = np.full((N, n + 1), X0, dtype=float)

    for i in range(n):
        dW = sqrt_dt * np.random.standard_normal(N)
        X[:, i + 1] = X[:, i] + mu(X[:, i]) * dt + sigma(X[:, i]) * dW

    return X

### Standard Euler–Maruyama

Same update rule, but increments are drawn **inside** the loop one step at a time:

$$\Delta W_i = \sqrt{\Delta t}\cdot Z_i, \quad Z_i \sim \mathcal{N}(0,1), \quad \text{shape: }(N,)$$

`sqrt_dt` is precomputed once outside the loop to avoid repeating `np.sqrt` on every iteration.


In [ ]:
mu    = lambda x: x
sigma = lambda x: np.ones_like(x)

# --- Benchmark ---
n_runs = 10
sim_kwargs = dict(mu=mu, sigma=sigma, n=10**4, N=10**4)

def timeit(fn, runs, **kwargs):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(**kwargs)
        times.append(time.perf_counter() - t0)
    return np.mean(times)

t_std = timeit(standard_euler_maruyama,   runs=n_runs, **sim_kwargs)
t_vec = timeit(vectorized_euler_maruyama, runs=n_runs, **sim_kwargs)

print(f"Standard   avg ({n_runs} runs): {t_std:.4f} s")
print(f"Vectorized avg ({n_runs} runs): {t_vec:.4f} s")
print(f"Speedup: {t_std / t_vec:.2f}x")

# --- Plot ---
X = vectorized_euler_maruyama(mu, sigma, n=500, N=300)
t_grid = np.linspace(0, 1, 501)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_grid, X[:30].T, alpha=0.5, linewidth=0.7)
ax1.set_xlabel("$t$")
ax1.set_ylabel("$X_t$")
ax1.set_title(r"Sample paths: $dX_t = X_t\,dt + dW_t$")

ax2.hist(X[:, -1], bins=50, density=True, color="steelblue", alpha=0.8)
ax2.set_xlabel("$X_1$")
ax2.set_ylabel("Density")
ax2.set_title("Terminal distribution $X_1$")

plt.tight_layout()
plt.show()

### Benchmark & Visualization

**Timing** — each function is timed over $n_\text{runs} = 10$ runs; mean wall-clock time is reported:

$$\bar{t} = \frac{1}{n_\text{runs}}\sum_{k=1}^{n_\text{runs}} t_k, \qquad \text{Speedup} = \frac{\bar{t}_\text{standard}}{\bar{t}_\text{vectorized}}$$

The vectorized version is faster because all $N \times n$ increments are generated in one NumPy call, eliminating $n$ Python-level calls to the random number generator inside the loop.

**Plots** use a lighter grid ($n=500$, $N=300$) for fast rendering:
- **Left** — 30 individual sample paths $t \mapsto X_t$
- **Right** — empirical density of the terminal value $X_1$